# Graph vs. Hypergraph Clustering — Reproducible Runner

This notebook runs the **fair, held-out, multi-seed** benchmark on your MITAB
datasets. Place each file at `<root>/<Dataset>/<Dataset>.txt` and set `SEARCH_ROOTS`.

It fixes the issues from the original manuscript: matched-k comparison, nested
λ/k selection on a validation split, ≥20 seeds, added baselines, randomized
controls, and statistics with effect sizes. Outputs are CSVs + PDF/PNG figures.


In [2]:
pip install -r requirements.txt

  Using cached matplotlib-3.9.4-cp39-cp39-win_amd64.whl.metadata (11 kB)
  Using cached python_igraph-1.0.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached leidenalg-0.12.0-cp38-abi3-win_amd64.whl.metadata (10 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following versions that require a different python version: 3.10.0 Requires-Python >=3.10; 3.10.0rc1 Requires-Python >=3.10; 3.10.1 Requires-Python >=3.10; 3.10.3 Requires-Python >=3.10; 3.10.5 Requires-Python >=3.10; 3.10.6 Requires-Python >=3.10; 3.10.7 Requires-Python >=3.10; 3.10.8 Requires-Python >=3.10; 3.10.9 Requires-Python >=3.10; 3.11.0 Requires-Python >=3.11; 3.11.0rc1 Requires-Python >=3.11; 3.11.0rc2 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement markov_clustering>=0.0.6 (from versions: 0.0.2.dev0, 0.0.3.dev0, 0.0.4.dev0, 0.0.5.dev0, 0.0.6.dev0)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for markov_clustering>=0.0.6


In [3]:
# If needed:
# !pip install numpy scipy pandas scikit-learn networkx matplotlib python-igraph leidenalg markov_clustering scikit-posthocs
import sys, os
sys.path.insert(0, os.path.abspath("."))   # so `import hgspectral` works
from hgspectral.config import Config
from hgspectral import pipeline


In [4]:
cfg = Config()
cfg.dataset_names = [
    "Cardiac", "BioCreative", "Affinomics", "Cancer", "Chromatin",
    "Coronavirus", "Cyanobacteria", "Diabetes", "Crohn's_disease",
]
cfg.search_roots = ["HG_Human2", "."]   # where <Dataset>/<Dataset>.txt live
cfg.n_seeds = 25                        # >=20 for publication
cfg.out_root = "fin_result"
cfg  # review the full config (lambda grid, split fractions, baselines, ...)


Config(dataset_names=['Cardiac', 'BioCreative', 'Affinomics', 'Cancer', 'Chromatin', 'Coronavirus', 'Cyanobacteria', 'Diabetes', "Crohn's_disease"], search_roots=['HG_Human1', '.'], top_k_taxids=3, min_complex_size=2, split_fracs=(0.6, 0.2, 0.2), stratify_by_size=True, split_seed=12345, lambda_grid=(0.0, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0), k_matched='rule', min_cluster_size=5, adaptive_criteria=('eigengap', 'silhouette', 'modularity'), kmax=15, n_seeds=25, base_seed=42, n_control_seeds=10, run_controls=True, run_louvain=True, run_leiden=True, run_mcl=True, q_threshold=0.05, min_term_size=5, out_root='fin_result')

In [24]:
cfg = Config()
cfg.dataset_names = [
    "Cardiac", "BioCreative", "Affinomics", "Cancer", "Chromatin",
    "Coronavirus", "Cyanobacteria", "Diabetes", "Crohn's_disease",
]
cfg.search_roots = ["HG_Human1", "."]   # where <Dataset>/<Dataset>.txt live
cfg.n_seeds = 25                        # >=20 for publication
cfg.out_root = "fin_result"
cfg  # review the full config (lambda grid, split fractions, baselines, ...)


Config(dataset_names=['Cancer'], search_roots=['HG_Human1', '.'], top_k_taxids=3, min_complex_size=2, split_fracs=(0.6, 0.2, 0.2), stratify_by_size=True, split_seed=12345, lambda_grid=(0.0, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0), k_matched='rule', min_cluster_size=5, adaptive_criteria=('eigengap', 'silhouette', 'modularity'), kmax=15, n_seeds=25, base_seed=42, n_control_seeds=10, run_controls=True, run_louvain=True, run_leiden=True, run_mcl=True, q_threshold=0.05, min_term_size=5, out_root='fin_result')

## Smoke test first (synthetic — NOT real)


In [6]:
'''
from scripts.make_synthetic_mitab import make
make("SynthA/SynthA.txt", seed=0)
smoke = Config(); smoke.dataset_names=["SynthA"]; smoke.search_roots=["."]
smoke.n_seeds=6; smoke.out_root="smoke_out"
pipeline.run_all(smoke)
'''

'\nfrom scripts.make_synthetic_mitab import make\nmake("SynthA/SynthA.txt", seed=0)\nsmoke = Config(); smoke.dataset_names=["SynthA"]; smoke.search_roots=["."]\nsmoke.n_seeds=6; smoke.out_root="smoke_out"\npipeline.run_all(smoke)\n'

In [7]:
pipeline.run_all(cfg)


[Cardiac] n_train=1152 k_matched=15 selected(lambda*=1.0, k*=15) val_bestF1=0.286
[BioCreative] n_train=327 k_matched=15 selected(lambda*=0.1, k*=15) val_bestF1=0.216
[Affinomics] n_train=326 k_matched=15 selected(lambda*=0.1, k*=15) val_bestF1=0.268
[Cancer] n_train=5481 k_matched=15 selected(lambda*=1.0, k*=14) val_bestF1=0.112
[ERROR] Cancer: ArpackNoConvergence: ARPACK error -1: No convergence (54811 iterations, 15/16 eigenvectors converged)
[Chromatin] n_train=1467 k_matched=15 selected(lambda*=10.0, k*=15) val_bestF1=0.245
[Coronavirus] n_train=3045 k_matched=15 selected(lambda*=0.1, k*=15) val_bestF1=0.201
[Cyanobacteria] n_train=198 k_matched=15 selected(lambda*=0.001, k*=15) val_bestF1=0.411
[Diabetes] n_train=658 k_matched=15 selected(lambda*=1.0, k*=15) val_bestF1=0.275
[ERROR] Diabetes: NetworkXAlgorithmError: Maximum number of swap attempts (134021000) exceeded before desired swaps achieved (1340210).
[Crohn's_disease] n_train=71 k_matched=14 selected(lambda*=1.0, k*=15) v

## Run the real benchmark
Each dataset writes `*_test_metrics.csv`, `*_controls.csv`,
`*_go_enrichment_summary.csv`, `*_manifest.csv` under `fin_result/<Dataset>/`.

## Aggregate, run statistics, render figures

In [10]:
import subprocess, sys
subprocess.run([sys.executable, "analyze_results.py",
                "--results", "fin_result/ALL_test_metrics.csv"], check=False)
import pandas as pd
pd.read_csv("fin_result/analysis/summary_by_method.csv").head(20)


,dataset,method,k_mode,n_seeds,pairwise_f1_mean,pairwise_f1_std,bestmatch_f1_sym_mean,bestmatch_f1_sym_std,b3_f1_mean,b3_f1_std,NMI_mean,NMI_std,ARI_mean,ARI_std,hyperedge_recovery_f1_mean,hyperedge_recovery_f1_std
0,Affinomics,graph_spectral,matched,25,0.043145,0.002545,0.189474,0.007991,0.044826,2.342246e-03,0.747517,1.356709e-02,0.186024,0.017722,0.170459,0.009253
1,Affinomics,graph_spectral_adaptive,adaptive,25,0.033759,0.011919,0.155892,0.044912,0.037175,1.222859e-02,0.667965,1.337574e-01,0.149522,0.056682,0.135213,0.046081
2,Affinomics,hyper_penalized,matched,25,0.044418,0.001452,0.200037,0.006626,0.049070,2.399315e-03,0.759740,1.037286e-02,0.200743,0.012517,0.184212,0.009003
3,Affinomics,hyper_penalized_adaptive,adaptive,25,0.005089,0.000467,0.052503,0.002773,0.011854,2.649021e-03,0.189539,3.503186e-02,0.005686,0.002316,0.032902,0.003643
4,Affinomics,hyper_zhou_lam0,matched,25,0.041248,0.004713,0.188065,0.010258,0.046393,2.492490e-03,0.744316,1.960817e-02,0.186001,0.021805,0.170165,0.010152
5,Affinomics,louvain,native,5,0.083696,0.001884,0.320270,0.005758,0.101806,1.996157e-03,0.843931,5.558155e-03,0.295062,0.020164,0.389342,0.008419
6,Affinomics,mcl,native,1,0.149644,0.000000,0.410418,0.000000,0.134989,0.000000e+00,0.896403,0.000000e+00,0.407829,0.000000,0.544143,0.000000
7,BioCreative,graph_spectral,matched,25,0.034416,0.002633,0.187707,0.008502,0.044554,3.239325e-03,0.760801,1.278545e-02,0.190058,0.021302,0.171288,0.008341
8,BioCreative,graph_spectral_adaptive,adaptive,25,0.021195,0.003790,0.125094,0.017812,0.025609,4.762803e-03,0.657527,4.981629e-02,0.114266,0.023624,0.105767,0.017587
9,BioCreative,hyper_penalized,matched,25,0.035054,0.001915,0.191946,0.006786,0.043678,3.345221e-03,0.758662,1.288361e-02,0.191224,0.019978,0.176094,0.009279


In [11]:
# Paired graph-vs-hypergraph tests (matched-k, held-out test), with effect sizes + CIs
pd.read_csv("fin_result/analysis/paired_graph_vs_hyper.csv")


,dataset,metric,hyper_mean,graph_mean,n,statistic,p_value,rank_biserial,median_diff,ci_mean_diff,ci_lo,ci_hi
0,Affinomics,bestmatch_f1_sym,0.200037,0.189474,25,40.0,4.894733e-04,0.753846,0.011777,0.010564,0.005823,0.015171
1,BioCreative,bestmatch_f1_sym,0.191946,0.187707,25,101.0,1.013974e-01,0.378462,0.001060,0.004239,-0.000050,0.008521
2,Cardiac,bestmatch_f1_sym,0.125748,0.099238,25,0.0,5.960464e-08,1.000000,0.026249,0.026509,0.023759,0.029029
3,Chromatin,bestmatch_f1_sym,0.242789,0.156657,25,0.0,5.960464e-08,1.000000,0.068076,0.086132,0.070044,0.101790
4,Coronavirus,bestmatch_f1_sym,0.243903,0.237714,25,42.0,6.313324e-04,0.741538,0.004865,0.006189,0.003272,0.009237
5,Crohn's_disease,bestmatch_f1_sym,0.543610,0.531969,25,31.0,1.398921e-04,0.809231,0.003855,0.011641,0.005772,0.017485
6,Cyanobacteria,bestmatch_f1_sym,0.493771,0.471219,25,0.0,5.960464e-08,1.000000,0.024790,0.022552,0.020321,0.024260


In [12]:
print(open("fin_result/analysis/friedman_nemenyi.txt").read())


Friedman p = 0.00380504077551137

Mean ranks (1=best):
method
graph_spectral     2.285714
hyper_zhou_lam0    2.714286
hyper_penalized    1.000000
dtype: float64



### Notes / knobs
- `cfg.lambda_grid` — λ sweep (includes 0 for the unregularized ablation).
- `cfg.stratify_by_size` — size-stratified hyperedge split (default True).
- `cfg.k_matched` — `"rule"` (n_train // min_cluster_size, capped) or an int.
- For weighting/min-size/species ablations, copy `cfg`, change one field, rerun
  into a different `out_root`, then compare the summary CSVs.
